# Special Char Removal

In [ ]:
import os
import re
from collections import defaultdict
import nltk

nltk.download('punkt')


def tokenize(text):
    """
    Convert text to lowercase and extract alphanumeric tokens.
    This simple tokenizer uses a regular expression to grab word characters.
    """
    text = text.lower()
    tokens = re.findall(r'\w+', text)
    return tokens

def is_valid_token(token):
    """
    Returns True if the token is not:
    - Purely numeric (like '0001', '989')
    - Starting with Greek or special Unicode chars
    """
    # Exclude numeric strings
    if token.isdigit():
        return False
    # Exclude Greek characters or special math symbols at the beginning
    if re.match(r"^[\u03B1-\u03C9\u03D0-\u03FF\u2100-\u214F]", token):
        return False
    return True

def build_inverted_index(folder_path):
    """
    Build an inverted index from all .txt files in the given folder.
    """
    inverted_index = defaultdict(set)
    doc_id = 1

    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue

            tokens = tokenize(text)
            for token in tokens:
                if is_valid_token(token):
                    inverted_index[token].add(doc_id)

            doc_id += 1

    return inverted_index

def print_inverted_index(inverted_index):
    """
    Print the inverted index in the specified format.
    """
    print("INVERTED INDEX")
    print("==============")
    print("Format: term: doc1, doc2, ...\n")

    for term in sorted(inverted_index.keys()):
        doc_ids = sorted(list(inverted_index[term]))
        doc_ids_str = ", ".join(str(doc_id) for doc_id in doc_ids)
        print(f"{term}: {doc_ids_str}")


### Stemming

In [1]:
import os
import re
from collections import defaultdict
from nltk.stem import PorterStemmer

# Initialize the stemmer globally
stemmer = PorterStemmer()

def tokenize(text):
    """
    Convert text to lowercase and extract alphanumeric tokens.
    Uses a regular expression to extract words (letters and digits).
    """
    text = text.lower()
    tokens = re.findall(r'\w+', text)
    return tokens

def is_valid_token(token):
    """
    Accept only tokens that:
    - Are made of ASCII characters
    - Do NOT contain any digits
    """
    return token.isascii() and not any(char.isdigit() for char in token)

def stem_token(token):
    """
    Apply stemming to a valid token using Porter Stemmer.
    """
    return stemmer.stem(token)

def get_doc_id_from_filename(filename):
    """
    Extract the numeric part from a filename like '1.txt' or '123.txt'.
    Returns an integer (e.g., 1, 123).
    Assumes the filename is strictly of the form '<number>.txt'.
    """
    name_part, ext = os.path.splitext(filename)
    return int(name_part)

def build_inverted_index(folder_path):
    """
    Build an inverted index from all .txt files in the folder,
    using the filename's numeric part as the doc_id (e.g. "7.txt" -> doc_id = 7).
    """
    inverted_index = defaultdict(set)
    files = sorted(f for f in os.listdir(folder_path) if f.endswith(".txt"))

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        doc_id = get_doc_id_from_filename(filename)

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue
        
        tokens = tokenize(text)

        for token in tokens:
            if is_valid_token(token):
                stemmed = stem_token(token)
                inverted_index[stemmed].add(doc_id)
    
    return inverted_index

def store_inverted_index(inverted_index, output_filename):
    """
    Write the inverted index into a file with the specified format:
    INVERTED INDEX
    ==============
    Format: term: doc1, doc2, ...
    """
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("INVERTED INDEX\n")
        f.write("==============\n")
        f.write("Format: term: doc1, doc2, ...\n\n")
        
        for term in sorted(inverted_index.keys()):
            doc_ids = sorted(list(inverted_index[term]))
            doc_ids_str = ", ".join(str(did) for did in doc_ids)
            f.write(f"{term}: {doc_ids_str}\n")
    
    print(f"Inverted index stored in '{output_filename}'.")

if __name__ == "__main__":
    folder_path = r'C:\Users\Public\Dev\Ph.D\2nd Semester\CS-675-IRS\Assignments\First\TXT 1-250'
    
    if not os.path.exists(folder_path):
        print("The folder path does not exist. Please check the path.")
    else:
        import nltk
        nltk.download('punkt')  # Only needed the first time
        inverted_index = build_inverted_index(folder_path)
        output_file = "inverted_index_stemmed.txt"
        store_inverted_index(inverted_index, output_file)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Inverted index stored in 'inverted_index_stemmed.txt'.


In [3]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')  # ✅ Correct tagger download

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [19]:
import os
import re
from collections import defaultdict
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag

# Download resources (only needed once)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

nltk.data.path.append(r'C:\nltk_data')  # ✅ Explicitly add correct path

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def tokenize(text):
    """
    Convert text to lowercase and extract alphanumeric tokens.
    """
    text = text.lower()
    tokens = re.findall(r'\w+', text)
    return tokens

def is_valid_token(token):
    """
    Accept only tokens that:
    - Are made of ASCII characters
    - Do NOT contain any digits
    """
    return token.isascii() and not any(char.isdigit() for char in token)

def get_wordnet_pos(treebank_tag):
    """
    Map NLTK POS tags to WordNet POS tags.
    """
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default fallback

def lemmatize_token(token, pos):
    """
    Lemmatize token using its POS tag.
    """
    return lemmatizer.lemmatize(token, pos)

def get_doc_id_from_filename(filename):
    """
    Extract numeric ID from filename like '123.txt'.
    """
    name_part, ext = os.path.splitext(filename)
    return int(name_part)

def build_inverted_index(folder_path):
    """
    Build inverted index from all .txt files using lemmatized tokens.
    """
    inverted_index = defaultdict(set)
    files = sorted(f for f in os.listdir(folder_path) if f.endswith(".txt"))

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        doc_id = get_doc_id_from_filename(filename)

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        tokens = tokenize(text)
        filtered_tokens = [token for token in tokens if is_valid_token(token)]

        # POS tagging for valid tokens
        tagged_tokens = pos_tag(filtered_tokens)

        for token, tag in tagged_tokens:
            lemma = lemmatize_token(token, get_wordnet_pos(tag))
            inverted_index[lemma].add(doc_id)

    return inverted_index

def store_inverted_index(inverted_index, output_filename):
    """
    Save the inverted index to a file in readable format.
    """
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("INVERTED INDEX\n")
        f.write("==============\n")
        f.write("Format: term: doc1, doc2, ...\n\n")
        
        for term in sorted(inverted_index.keys()):
            doc_ids = sorted(list(inverted_index[term]))
            doc_ids_str = ", ".join(str(did) for did in doc_ids)
            f.write(f"{term}: {doc_ids_str}\n")

    print(f"Inverted index stored in '{output_filename}'.")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [20]:

if __name__ == "__main__":
    folder_path = r'C:\Users\Public\Dev\Ph.D\2nd Semester\CS-675-IRS\Assignments\First\TXT 1-250'

    if not os.path.exists(folder_path):
        print("The folder path does not exist. Please check the path.")
    else:
        inverted_index = build_inverted_index(folder_path)
        output_file = "inverted_index_lemm.txt"
        store_inverted_index(inverted_index, output_file)


Inverted index stored in 'inverted_index_lemm.txt'.


In [14]:
import nltk
nltk.data.path.append(r'C:\nltk_data')  # ✅ Explicitly add correct path